# PMT saturation with LED pulses

One LED lights both PMTs. **PMT A** (Ch1) is the device under test. **PMT B** (Ch2) gets far less light and is
the linear reference. While B is linear, its charge is proportional to the true light per pulse, so A's
departure from proportionality to B is A's non-linearity. LED non-linearity and drift cancel.

**Two regimes** (set `REGIME`):
- **`"short"`:** ~100 ns pulses probe the **space-charge** limit on peak anode current.
- **`"long"`:** 10 µs pulses probe **charge depletion** of the base capacitors. Depletion shows as the current
  drooping during the pulse.

**Rules:**
- **Pulse rate below 5 Hz,** so the bases recharge between pulses.
- **HV is checked, never changed.** Bias with `python -m ad3pulse hv set ...` first.
- **Each point is auto-ranged** (V/div per PMT) from a few probe frames.
- **Peak guard:** if a PMT pulse exceeds `MAX_PMT_PEAK_V` at the 50 Ω input, the LED stops and the sweep ends.
- **PMT B has fewer base capacitors** and may deplete first. In the long regime, a *rising* A(t)/B(t) late in the
  pulse, or a change in B's late/early ratio, means B is no longer a clean reference. Stop and consider a base
  with more capacitors for B.

Raw frames are saved to `runs/saturation/`. All analysis below can be re-run from the saved files.

In [ ]:
import numpy as np

REGIME = "short"                         # "short" (100 ns, space charge) or "long" (10 us, depletion)

# ---- Instruments -----------------------------------------------------------------
SCOPE_RESOURCE = "192.168.0.101"
HV_RESOURCE = "192.168.0.100"
HV_EXPECTED_V = {0: 1550.0, 1: 875.0}   # HV channel -> expected |V|; checked, never changed
HV_TOLERANCE_V = 2.0

PMTS = {                                 # Rigol channel -> label
    1: "PMT A (HV ch0, -1550 V)",
    2: "PMT B (HV ch1, -875 V)",
}
TEST_CH, REF_CH = 1, 2                   # A under test, B as the linear reference
Q1_PC = {1: 0.949, 2: 0.492}             # SPE charges at these HV (results/spe_2026-10-07)
PULSER_CHANNEL = 3                       # tee'd copy of the LED drive; trigger source

# ---- LED pulses (AD3 W1) -----------------------------------------------------------
PULSE_RATE_HZ = 2.0
assert PULSE_RATE_HZ < 5, "keep the rate below 5 Hz so the bases recharge between pulses"
MAX_LED_V = 3.0                          # hard limit on the AD3 amplitude

# ---- Scope safety and ranging --------------------------------------------------------
MAX_PMT_PEAK_V = 3.0                     # |V| limit at the 50 Ohm inputs (rated 5 Vrms, datasheet p. 11;
                                         # no transient overvoltage allowed): user's choice, 2026-10-07
PMT_START_SCALE_V_DIV = 0.01
PMT_OFFSET_DIV = 3.0                     # trace shifted up: 7 of 8 divisions for negative pulses
SCALE_LADDER = (0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0)
PROBE_FRAMES = 10                        # per auto-range step (not saved)
PULSER_SCALE_V_DIV = 0.5
PULSER_OFFSET_V = -1.5
TRIG_LEVEL_V = 0.5

# ---- Regimes (times in seconds from the trigger) -------------------------------------
REGIMES = {
    "short": dict(
        width_s=100e-9,
        timebase_s_div=100e-9, time_offset_s=0.0, memory_depth="1k",   # -500..+500 ns, 1 GS/s
        baseline=(-500e-9, -300e-9),
        signal=(40e-9, 440e-9),          # light ~80-180 ns; extra room for broadened pulses
        pulser_window=(20e-9, 80e-9),
        time_bins=None, early=None, late=None,
        timing_amplitude=1.9,
        amplitudes=np.round(1.55 + np.geomspace(0.10, 1.45, 16), 3),
    ),
    "long": dict(
        width_s=10e-6,
        timebase_s_div=5e-6, time_offset_s=10e-6, memory_depth="10k",  # ~-15..+35 us, 200 MS/s
        baseline=(-14e-6, -2e-6),
        signal=(0.0, 14e-6),             # the 10 us pulse plus its tail
        post=(14e-6, 34e-6),             # recovery / overshoot after the pulse
        pulser_window=(1e-6, 9e-6),
        time_bins=np.arange(0.0, 10.01e-6, 1e-6),   # charge per 1 us across the pulse
        early=slice(1, 3), late=slice(7, 9),        # bins for late/early ratios
        timing_amplitude=1.7,
        amplitudes=np.round(1.55 + np.geomspace(0.02, 1.45, 16), 3),
    ),
}
AMPLITUDES_V = None                      # up-sweep list; None = the regime's default. The down-sweep
                                         # revisits every DOWN_EVERY-th point.
DOWN_EVERY = 2
BASELINE_PER_FRAME = False               # one median baseline per run (adds no noise)

# ---- Run sizes ---------------------------------------------------------------------
REACH_FRAMES = 30
# Low-light statistics for the A/B slope (lowlight cell). A is far from saturation here, so the
# user allowed 10 Hz (2026-10-07); the < 5 Hz rule applies to the saturating points.
LOWLIGHT_AMPLITUDES_V = [1.841, 1.898, 1.966, 2.048]
LOWLIGHT_FRAMES = 1000
LOWLIGHT_RATE_HZ = 10.0
assert LOWLIGHT_RATE_HZ <= 10
SCAN_FRAMES = 200
TIMING_FRAMES = 20
OUTPUT_DIR = "runs/saturation"
SCOPE_RETRIES = 3

# ---- Analysis -----------------------------------------------------------------------
REF_MIN_NPE = 3.0                        # lowest B signal used to fix the A/B slope
FIT_POINTS = 4                           # number of lowest usable points in that fit
REF_SHAPE_TOLERANCE = 0.02               # flag B if its late/early ratio moves by more (and > 3 sigma)
REF_FWHM_TOLERANCE = 0.05                # short regime: flag B if its mean-pulse FWHM moves by more

## Helpers (no hardware)
Per-point summaries from raw frames: charge, peak current, binned charge and mean waveforms.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from ad3pulse.hardware import FrameSet
from ad3pulse.saturation import (binned_charge, crossing, flag_reference, fwhm, late_early,
                                 linearity, peak_current, shape_ratio, summarize_point)
from ad3pulse.spe import baseline_subtract, integrate_charge

R = REGIMES[REGIME]
if AMPLITUDES_V is None:
    AMPLITUDES_V = R["amplitudes"]
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)


def summarize(fs):
    """Summary of one saved point: per-PMT charge/peak means, mean waveform, binned charge."""
    t = fs.time_s
    row = {"amplitude_V": fs.meta["amplitude_V"], "label": fs.meta["label"],
           "pulser_height_V": fs.meta.get("pulser_height_V"), "t": t, "ch": {}}
    for ch in PMTS:
        if ch not in fs.codes:
            continue
        v = baseline_subtract(fs.volts(ch), t, R["baseline"], BASELINE_PER_FRAME)
        q = integrate_charge(v, t, R["signal"])
        pk = peak_current(v, t, R["signal"], smooth=5)
        mean_v = v.mean(axis=0)
        c = {"summary": summarize_point(q, pk, fs.meta["amplitude_V"]), "mean_v": mean_v,
             "fwhm_s": fwhm(t, mean_v)}
        c["npe"] = c["summary"].q_mean_pC / Q1_PC[ch]
        if R["time_bins"] is not None:
            c["bins"] = binned_charge(v, t, R["time_bins"])
            c["late_early"], c["late_early_err"] = late_early(c["bins"], R["early"], R["late"])
            c["q_post_pC"] = float(integrate_charge(v, t, R["post"]).mean())
        row["ch"][ch] = c
    return row


def describe(row):
    a, b = row["ch"][TEST_CH], row["ch"][REF_CH]
    text = (f"LED {row['amplitude_V']:.3f} V: A {a['npe']:8.1f} PE ({a['summary'].peak_mean_mA:6.2f} mA), "
            f"B {b['npe']:7.2f} PE ({b['summary'].peak_mean_mA:6.3f} mA)")
    if b["npe"] > 0:
        text += f", A/B {a['npe'] / b['npe']:7.1f}"
    if "late_early" in b:
        text += (f", late/early A {a['late_early']:.3f}±{a['late_early_err']:.3f} "
                 f"B {b['late_early']:.3f}±{b['late_early_err']:.3f}")
    else:
        text += f", FWHM A {a['fwhm_s'] * 1e9:.0f} ns B {b['fwhm_s'] * 1e9:.0f} ns"
    return text

## Connect
Checks the HV, then opens the scope (through `ScopeSession`) and the AD3.

In [ ]:
import time
from datetime import datetime

from ad3pulse.acquire import PeakLimitExceeded, ScopeSession, ScopeSettings, autorange, check_peaks
from ad3pulse.hardware import IsegHV
from ad3pulse.instrument import AnalogDiscovery, PulseSpec

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# HV: check only.
with IsegHV(HV_RESOURCE) as hv:
    hv_state = {}
    for hv_ch, expected in HV_EXPECTED_V.items():
        hv.channel = hv_ch
        status, v, i = hv.read_status(), hv.read_voltage(), hv.read_current()
        hv_state[hv_ch] = {"V": v, "I_A": i}
        print(f"HV ch{hv_ch}: {v:.2f} V, {i * 1e6:.2f} uA, {', '.join(hv.describe_status(status))}")
        if not status & IsegHV.IS_ON or abs(abs(v) - expected) > HV_TOLERANCE_V:
            raise RuntimeError(f"HV ch{hv_ch} is not on at {expected:g} V; set it with "
                               f"'python -m ad3pulse hv set {expected:g} --channel {hv_ch}'")

ad3 = AnalogDiscovery().open()
PULSE = PulseSpec(amplitude=0.5, width=R["width_s"], pre_delay=100e-9, post_delay=1e-6)

settings = ScopeSettings(
    pmt_scale_v_div={ch: PMT_START_SCALE_V_DIV for ch in PMTS}, pulser_channel=PULSER_CHANNEL,
    pulser_scale_v_div=PULSER_SCALE_V_DIV, pulser_offset_v=PULSER_OFFSET_V,
    pmt_offset_div=PMT_OFFSET_DIV, timebase_s_div=R["timebase_s_div"],
    time_offset_s=R["time_offset_s"], memory_depth=R["memory_depth"], trig_level_v=TRIG_LEVEL_V)
session = ScopeSession(
    SCOPE_RESOURCE, settings, ad3, rate_hz=PULSE_RATE_HZ, out_dir=out_dir, stamp=stamp,
    max_led_v=MAX_LED_V, retries=SCOPE_RETRIES, pulser_window=R["pulser_window"],
    run_meta={"regime": REGIME, "width_s": R["width_s"], "hv": hv_state, "q1_pC": Q1_PC}).open()
print(f"Scope: {session.scope.idn}")
for ch in PMTS:
    print(f"  Ch{ch}: V/div, offset as read back = {session.channel_setting(ch)}")

per_point_s = PROBE_FRAMES / PULSE_RATE_HZ * 2 + SCAN_FRAMES / PULSE_RATE_HZ
n_points = len(AMPLITUDES_V) + len(AMPLITUDES_V[-2::-DOWN_EVERY])
print(f"{REGIME} regime, {R['width_s'] * 1e9:g} ns pulses at {PULSE_RATE_HZ:g} Hz; full scan of "
      f"{n_points} points needs >= {n_points * per_point_s / 60:.0f} min plus readout")


def take_point(amplitude, n_frames, label):
    """Auto-range, record, save and summarize one point (raises PeakLimitExceeded)."""
    autorange(session, amplitude, R["baseline"], MAX_PMT_PEAK_V, n_probe=PROBE_FRAMES,
              ladder=SCALE_LADDER)
    fs = session.acquire(amplitude, n_frames, label)
    check_peaks(fs, session.settings, R["baseline"], MAX_PMT_PEAK_V)
    return summarize(fs)

## Timing check
Mean waveforms at a moderate setting, normalised, with the analysis windows shaded. Check that the signal
window contains the whole pulse (and, for the long regime, that the 1 µs bins cover the light).

In [ ]:
ad3.start_pulse_train(PULSE, PULSE_RATE_HZ)
try:
    timing_row = take_point(R["timing_amplitude"], TIMING_FRAMES, f"{REGIME}_timing")
finally:
    ad3.stop_output(0)
print(describe(timing_row))

fig, ax = plt.subplots(figsize=(9, 4))
t_us = timing_row["t"] * 1e6
for ch, c in timing_row["ch"].items():
    ax.plot(t_us, c["mean_v"] / abs(c["mean_v"].min()), lw=1, label=PMTS[ch])
spans = [(R["signal"], "C2"), (R["baseline"], "C0")] + ([(R["post"], "0.6")] if "post" in R else [])
for (lo, hi), color in spans:
    ax.axvspan(lo * 1e6, hi * 1e6, color=color, alpha=0.12)
if R["time_bins"] is not None:
    for e in R["time_bins"]:
        ax.axvline(e * 1e6, color="0.7", lw=0.5)
ax.set_xlabel("time from trigger (µs)")
ax.set_ylabel("mean / |peak|")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(out_dir / f"{REGIME}_timing_{stamp}.png", dpi=150)
plt.show()

## Reach scan
A quick ramp (`REACH_FRAMES` per point) up the amplitude list, with the peak guard. It shows where B has a usable
signal and where A bends. Trim `AMPLITUDES_V` from this before the full scan.

In [ ]:
reach = []
ad3.start_pulse_train(PULSE, PULSE_RATE_HZ)
try:
    for amp in AMPLITUDES_V:
        try:
            row = take_point(float(amp), REACH_FRAMES, f"{REGIME}_reach")
        except PeakLimitExceeded as exc:
            print(f"LED {amp:.3f} V: PEAK GUARD: {exc}; stopping the ramp")
            break
        reach.append(row)
        print(describe(row), f"[V/div {session.settings.pmt_scale_v_div}]")
finally:
    ad3.stop_output(0)

## Low-light statistics
`LOWLIGHT_FRAMES` pulses at each of `LOWLIGHT_AMPLITUDES_V`, at `LOWLIGHT_RATE_HZ`. A is still linear here and
B has only ~0.5-6 PE per pulse, so many pulses are needed to fix the A/B slope.

In [ ]:
lowlight = []
session.rate_hz = LOWLIGHT_RATE_HZ
ad3.start_pulse_train(PULSE, LOWLIGHT_RATE_HZ)
try:
    for amp in LOWLIGHT_AMPLITUDES_V:
        t0 = time.monotonic()
        lowlight.append(take_point(float(amp), LOWLIGHT_FRAMES, f"{REGIME}_low"))
        print(describe(lowlight[-1]), f"[{time.monotonic() - t0:.0f} s]", flush=True)
finally:
    ad3.stop_output(0)
    session.rate_hz = PULSE_RATE_HZ

## Full scan
Low → high (every point), then high → low (every `DOWN_EVERY`-th point), so drift and hysteresis show up. The
down-sweep starts below the point where the guard tripped, if it did.

In [ ]:
points = []
ad3.start_pulse_train(PULSE, PULSE_RATE_HZ)
try:
    top = len(AMPLITUDES_V)
    for k, amp in enumerate(AMPLITUDES_V):
        try:
            points.append(("up", take_point(float(amp), SCAN_FRAMES, f"{REGIME}_up")))
        except PeakLimitExceeded as exc:
            print(f"LED {amp:.3f} V: PEAK GUARD: {exc}; ending the up-sweep")
            top = k
            break
        print("up  ", describe(points[-1][1]), flush=True)
    for amp in AMPLITUDES_V[:top][-2::-DOWN_EVERY]:
        points.append(("down", take_point(float(amp), SCAN_FRAMES, f"{REGIME}_down")))
        print("down", describe(points[-1][1]), flush=True)
finally:
    ad3.stop_output(0)

## Optional: load a saved scan instead (no hardware needed)
Skip this after a live scan.

In [ ]:
LOAD_STAMP = None                        # e.g. "20261008_101500"; None = newest scan of this REGIME
files = sorted(out_dir.glob(f"{REGIME}_up_*.npz")) + sorted(out_dir.glob(f"{REGIME}_down_*.npz"))
if not files:
    raise FileNotFoundError(f"no {REGIME}_up/down files in {out_dir}")
run = LOAD_STAMP or max("_".join(f.stem.split("_")[-2:]) for f in files)
points = []
for f in files:
    if f.stem.endswith(run):
        row = summarize(FrameSet.load(f))
        points.append(("up" if "_up_" in f.name else "down", row))
points.sort(key=lambda p: (p[0] == "down", p[1]["amplitude_V"] * (-1 if p[0] == "down" else 1)))
stamp = f"reanalysis_{run}"
print(f"{REGIME} run {run}: {len(points)} points")
for sweep, row in points:
    print(f"{sweep:4s}", describe(row))

## Analysis
- **Slope:** the A/B slope comes from the `FIT_POINTS` lowest points with B ≥ `REF_MIN_NPE` photoelectrons.
- **Deviation:** A / (k·B) − 1. Points where B's own shape changed are marked with ×; A's curve is unreliable
  there.
- **Long regime:** A(t)/B(t) per 1 µs bin, normalised to the slope-fit points. A falling curve means A is
  depleting; a rising curve means B is.

In [ ]:
sweep = np.array([p[0] for p in points])
rows = [p[1] for p in points]
amp = np.array([r["amplitude_V"] for r in rows])
S = {ch: [r["ch"][ch]["summary"] for r in rows] for ch in (TEST_CH, REF_CH)}
nA = np.array([s.q_mean_pC for s in S[TEST_CH]]) / Q1_PC[TEST_CH]
nA_err = np.array([s.q_err_pC for s in S[TEST_CH]]) / Q1_PC[TEST_CH]
nB = np.array([s.q_mean_pC for s in S[REF_CH]]) / Q1_PC[REF_CH]
nB_err = np.array([s.q_err_pC for s in S[REF_CH]]) / Q1_PC[REF_CH]
peakA = np.array([s.peak_mean_mA for s in S[TEST_CH]])
qA = np.array([s.q_mean_pC for s in S[TEST_CH]])

usable = np.flatnonzero((nB >= REF_MIN_NPE) & (sweep == "up"))
fit_idx = usable[np.argsort(nB[usable])][:FIT_POINTS]
if len(fit_idx) < 2:
    raise RuntimeError(f"fewer than 2 up-sweep points with B >= {REF_MIN_NPE} PE; lower REF_MIN_NPE "
                       "or extend AMPLITUDES_V")
fit_range = (nB[fit_idx].min(), nB[fit_idx].max())
lin = linearity(nB, nA, fit_range, ref_err=nB_err, test_err=nA_err)
print(f"A/B light ratio (in PE) k = {lin.k:.2f} ± {lin.k_err:.2f}, from B = "
      f"{fit_range[0]:.1f}..{fit_range[1]:.1f} PE")

# Reference self-check: B's pulse shape must not change with intensity.
if R["time_bins"] is not None:
    b_shape = np.array([r["ch"][REF_CH]["late_early"] for r in rows])
    b_shape_err = np.array([r["ch"][REF_CH]["late_early_err"] for r in rows])
    b_shape_name = "B late/early"
    b_flag = flag_reference(b_shape, b_shape_err, fit_idx, REF_SHAPE_TOLERANCE)
else:
    b_shape = np.array([r["ch"][REF_CH]["fwhm_s"] for r in rows])
    b_shape_err = None
    b_shape_name = "B FWHM"
    b_flag = flag_reference(b_shape, None, fit_idx, REF_FWHM_TOLERANCE)
b_flag &= nB >= REF_MIN_NPE
if b_flag.any():
    print(f"WARNING: {b_shape_name} moved > {REF_SHAPE_TOLERANCE:.0%} at LED "
          f"{', '.join(f'{a:.3f}' for a in amp[b_flag])} V: B may be saturating (or the LED shape changed)")

up = sweep == "up"
good = up & ~b_flag & (nB >= REF_MIN_NPE)   # points where B is a trustworthy reference
xs = {"N_pe (A)": nA, "peak current A (mA)" if REGIME == "short" else "charge A (nC)":
      peakA if REGIME == "short" else qA / 1e3}
for name, x in xs.items():
    for level in (0.05, 0.10):
        xc = crossing(x[good], lin.deviation[good], level)
        print(f"  {level:.0%} low at {name} = {xc:.4g}" if xc is not None
              else f"  never {level:.0%} low within the scan ({name})")

fig, axes = plt.subplots(1, 1 + len(xs), figsize=(5 * (1 + len(xs)), 4.2))
ax = axes[0]
for sw, m in (("up", "o"), ("down", "s")):
    sel = sweep == sw
    ax.errorbar(nB[sel], nA[sel], xerr=nB_err[sel], yerr=nA_err[sel], fmt=m, ms=4, capsize=2, label=sw)
xx = np.geomspace(max(nB[nB > 0].min(), 1e-3), nB.max(), 50)
ax.plot(xx, lin.k * xx, "k--", lw=0.8, label=f"linear, k = {lin.k:.1f}")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("B (PE per pulse)"); ax.set_ylabel("A (PE per pulse)")
ax.set_title(f"{REGIME} regime: A against the B reference", fontsize=9)
for ax, (name, x) in zip(axes[1:], xs.items()):
    for sw, m in (("up", "o"), ("down", "s")):
        sel = (sweep == sw) & ~b_flag & (nB >= REF_MIN_NPE)
        ax.errorbar(x[sel], 100 * lin.deviation[sel], yerr=100 * lin.deviation_err[sel], fmt=m, ms=4,
                    capsize=2, label=sw)
    if b_flag.any():
        ax.plot(x[b_flag], 100 * lin.deviation[b_flag], "x", color="C3", label="B suspect")
    for level in (-5, -10):
        ax.axhline(level, color="0.6", ls=":", lw=0.8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_xscale("log")
    ax.set_xlabel(name); ax.set_ylabel("deviation from linear (%)")
for ax in axes:
    ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(out_dir / f"{REGIME}_linearity_{stamp}.png", dpi=150)
plt.show()

# Pulse shapes and the reference self-check.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
cmap = plt.get_cmap("viridis")
order = np.argsort(nA)
t_us = rows[0]["t"] * 1e6
for ax, ch in zip(axes[:2], (TEST_CH, REF_CH)):
    for j, i in enumerate(order):
        if sweep[i] != "up":
            continue
        mv = rows[i]["ch"][ch]["mean_v"]
        if R["time_bins"] is not None:   # long pulses: smooth over ~50 ns for display
            n_s = max(1, int(50e-9 / (t_us[1] - t_us[0]) / 1e-6))
            mv = np.convolve(mv, np.ones(n_s) / n_s, mode="same")
        ax.plot(t_us, mv / abs(mv.min()), lw=0.8, color=cmap(j / max(len(order) - 1, 1)))
    ax.set_xlim(R["signal"][0] * 1e6 - 0.05, R["signal"][1] * 1e6)
    ax.set_xlabel("time from trigger (µs)"); ax.set_ylabel("mean / |peak|")
    ax.set_title(f"{PMTS[ch]}, colour = intensity", fontsize=9)
ax = axes[2]
if R["time_bins"] is not None:
    ref_ratio = np.mean([shape_ratio(rows[i]["ch"][TEST_CH]["bins"], rows[i]["ch"][REF_CH]["bins"])
                         for i in fit_idx], axis=0)
    centers = 0.5 * (R["time_bins"][1:] + R["time_bins"][:-1]) * 1e6
    for j, i in enumerate(order):
        if sweep[i] != "up" or nB[i] < REF_MIN_NPE:
            continue
        r = shape_ratio(rows[i]["ch"][TEST_CH]["bins"], rows[i]["ch"][REF_CH]["bins"]) / ref_ratio
        ax.plot(centers, r, marker=".", lw=0.8, color=cmap(j / max(len(order) - 1, 1)),
                label=f"A {nA[i]:.0f} PE" if j % 3 == 0 else None)
    ax.axhline(1, color="k", lw=0.6)
    ax.set_xlabel("time bin centre (µs)")
    ax.set_ylabel("A(t)/B(t), normalised to low light")
    ax.set_title("falls late: A depleting; rises late: B depleting", fontsize=9)
    ax.legend(fontsize=7)
else:
    ax.plot(nB, b_shape * 1e9, "o", ms=4)
    ax.plot(nB[b_flag], b_shape[b_flag] * 1e9, "x", color="C3", label="flagged")
    ax.set_xscale("log")
    ax.set_xlabel("B (PE per pulse)"); ax.set_ylabel("B mean-pulse FWHM (ns)")
    ax.set_title("reference self-check", fontsize=9)
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(out_dir / f"{REGIME}_shapes_{stamp}.png", dpi=150)
plt.show()

if R["time_bins"] is not None:
    fig, ax = plt.subplots(figsize=(6, 4))
    a_shape = np.array([r["ch"][TEST_CH]["late_early"] for r in rows])
    a_shape_err = np.array([r["ch"][TEST_CH]["late_early_err"] for r in rows])
    qB = np.array([s.q_mean_pC for s in S[REF_CH]])
    for q, sh, er, m, name in ((qA, a_shape, a_shape_err, "o", "A"), (qB, b_shape, b_shape_err, "s", "B")):
        w = 1 / er[fit_idx] ** 2
        norm = np.sum(w * sh[fit_idx]) / np.sum(w)   # low-light value from the slope-fit points
        sel = q > 0
        ax.errorbar(q[sel] / 1e3, sh[sel] / norm, yerr=er[sel] / norm, fmt=m, ms=4, capsize=2, label=name)
    ax.axhspan(1 - REF_SHAPE_TOLERANCE, 1 + REF_SHAPE_TOLERANCE, color="C2", alpha=0.15)
    ax.set_xscale("log")
    ax.set_xlabel("charge per pulse (nC)"); ax.set_ylabel("late/early, normalised to low light")
    ax.set_title("droop against each PMT's own charge", fontsize=9)
    ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out_dir / f"{REGIME}_droop_{stamp}.png", dpi=150)
    plt.show()

## Close

In [ ]:
ad3.stop_output(0)
ad3.close()
session.close()

## Emergency: LED pulses off
Stops the AD3 output even if the cells above were not run. HV is not touched.

In [ ]:
try:
    ad3.stop_output(0)
    ad3.close()
except NameError:
    from ad3pulse.instrument import AnalogDiscovery
    with AnalogDiscovery() as _dev:
        _dev.stop_output(0)
print("AD3 output stopped")